# Lab 09 — From Tables/Text to JSONL for LLM Workloads
### Week 2 · Data Engineering for LLM Pipelines

The tabular work from Labs 05–08 gets you clean, validated records. But an LLM pipeline
doesn't consume a DataFrame — it consumes **JSONL**: one JSON object per line, streamable,
append-friendly, line-recoverable. In this lab you take an **LLM-native corpus** (help
articles, policies, release notes, FAQs), pull it across a **service boundary** (a local
API), **clean / filter / dedupe** it under governance rules, and emit two JSONL products:
**RAG chunks** (with provenance) and **SFT/eval rows** (`input`/`output`/`metadata`).

**By the end you will be able to:**
1. Say why **JSONL** beats a single JSON array for large corpora (streaming, append,
   line-recoverable, map-reduce-friendly).
2. Serve a corpus over a small **FastAPI** service and **ingest it with pagination**.
3. Apply a cleaning pipeline: **governance** + **language** filters, missing/short-text
   handling, and **content-signature deduplication** (not `doc_id`).
4. Write **RAG-chunk JSONL** (`doc_id → chunk_id`, with metadata) and **SFT JSONL**, then
   **validate line-by-line** and emit a reviewer sample.

> **Hints stay light (Labs 05–08).** Each Part opens with a **Toolbox**; you assemble the
> pieces. Target **26/26**; a red check never halts the notebook.
>
> ⚠️ **CURRENCY FLAG — FastAPI startup.** The classic `@app.on_event("startup")` hook is
> **deprecated**; this lab uses the modern **`lifespan`** context manager and stores state
> on `app.state`. We also consume the API the production way — a **background `uvicorn`
> server + `requests`** — rather than the `TestClient`, whose `httpx` backend is itself
> mid-migration on current Starlette.


## Setup — imports, folders, and the `check()` helper

In [ ]:
%pip install -r requirements.txt

In [ ]:

import csv, random, re, hashlib, json, time, threading
from pathlib import Path
from datetime import datetime, timedelta
from contextlib import asynccontextmanager

import numpy as np
import pandas as pd
import orjson
import requests
import uvicorn
from fastapi import FastAPI, HTTPException

for p in ["artifacts/jsonl", "artifacts/samples", "tools", "data"]:
    Path(p).mkdir(parents=True, exist_ok=True)

print("pandas", pd.__version__, "| ready")

In [ ]:

# ── soft self-check: prints PASS/FAIL, never raises ──────────────────────────
_score = {"pass": 0, "fail": 0}
def check(label, predicate):
    try:
        ok = bool(predicate() if callable(predicate) else predicate); note = ""
    except Exception as e:
        ok, note = False, f"  [error: {type(e).__name__}: {e}]"
    _score["pass" if ok else "fail"] += 1
    print(f"{'\u2705 PASS' if ok else '\u274c FAIL'} \u2014 {label}{note}")
def score():
    t = _score["pass"] + _score["fail"]
    print(f"\n{'='*46}\n  {_score['pass']}/{t} checks passing  ({_score['fail']} to go)\n{'='*46}")

check("Setup: artifact folders exist", lambda: all(Path(p).is_dir() for p in ["artifacts/jsonl","artifacts/samples","data"]))


---
## Part A — Build the LLM-native corpus

A realistic corpus is metadata-rich (for RAG filtering) and deliberately *messy*: a few
empty bodies and a cluster of exact-duplicate content under different `doc_id`s. The
builder below is **provided** — your job starts at characterizing it.

In [ ]:

def build_corpus_csv(path="data/corpus_llm.csv", n=1000, seed=42):
    """Synthetic LLM corpus: docs/policies/release-notes/FAQs, with planted dups + empties."""
    rng = random.Random(seed)
    TYPES = ["help_article","policy","release_note","faq"]
    SECTIONS = ["Overview","Setup","Troubleshooting","FAQ"]
    TAGS = ["billing","security","compliance","sso","api","governance","export","retention","privacy","rate_limits"]
    LANGS = ["en","en","en","de","fr"]        # mostly English
    CONF = ["public","internal"]
    now = datetime(2025, 2, 10)
    boiler = ("This article explains how to configure single sign-on with step-by-step instructions. "
              "Use the admin console to enable SAML and verify claim mappings. "
              "Common pitfalls include clock skew and incorrect audience URIs. ")
    rows = []
    for i in range(1, n + 1):
        kind = rng.choice(TYPES); doc_id = f"DOC-{i:04d}"
        title = {
            "help_article": f"How to configure SSO (v{rng.randint(1,5)}).",
            "policy": f"Data Retention Policy \u2014 Region {rng.choice(['US','EU','APAC'])}",
            "release_note": f"Release 2025{rng.randint(1,12):02d} \u2014 Key fixes",
            "faq": f"FAQ: {rng.choice(['Exports','Rate Limits','Privacy','Billing'])}",
        }[kind]
        body = (boiler * rng.randint(1,3)) + f"Additional details about {rng.choice(TAGS)} and {rng.choice(TAGS)}. Ref {doc_id}."
        rows.append({
            "doc_id": doc_id, "type": kind, "title": title, "section": rng.choice(SECTIONS),
            "body_text": body, "tags": ",".join(rng.sample(TAGS, k=rng.randint(2,4))),
            "source_url": f"https://example.local/{kind}/{doc_id.lower()}",
            "created_at": (now - timedelta(days=rng.randint(0,240))).strftime('%Y-%m-%d'),
            "updated_at": (now - timedelta(days=rng.randint(0,30))).strftime('%Y-%m-%d'),
            "language": rng.choice(LANGS), "confidentiality": rng.choice(CONF),
        })
    rows[0].update({"confidentiality": "public", "language": "en"})   # canonical, survives filters
    for j in range(9):                                                # 9 exact-content duplicates
        rows.append({**rows[0], "doc_id": f"DOC-DUP-{j:02d}"})
    for k in range(5):                                                # 5 empty bodies to drop
        rows.append({**rows[0], "doc_id": f"DOC-EMPTY-{k:02d}", "body_text": ""})
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)
    return len(rows)

n_written = build_corpus_csv()
raw = pd.read_csv("data/corpus_llm.csv")
print("wrote", n_written, "rows | body_text dtype:", raw["body_text"].dtype)
raw.head(3)


### A1 — Characterize the corpus

Write `characterize(df)` → a dict with `n_rows`, `n_by_type` (a `{type: count}` dict),
`n_empty_body` (blank or missing `body_text`), `n_public`, and `n_en`. You'll use these
numbers to sanity-check the funnel later.

> **🧰 Toolbox for Part A** — `len(df)` · `df["type"].value_counts().to_dict()` ·
> `Series.fillna("")` · `Series.str.len()` · boolean masks · `(mask).sum()`.
> ⚠️ In pandas 3.0 a CSV's text columns load as the **`str`** dtype; blanks read back as
> empty strings or `NaN`, so guard both when counting "empty".

In [ ]:

def characterize(df):
    # TODO: return {n_rows, n_by_type (dict), n_empty_body, n_public, n_en}.
    #       Guard empty AND NaN when counting blank bodies.
    return {}

stats = characterize(raw)
stats

In [ ]:

check("A1: corpus has 1,014 rows (1,000 + 9 dups + 5 empties)", lambda: stats.get("n_rows") == 1014)
check("A1: exactly 5 empty bodies detected", lambda: stats.get("n_empty_body") == 5)
check("A1: public / en counts match the frame",
      lambda: stats.get("n_public") == int((raw["confidentiality"]=="public").sum())
              and stats.get("n_en") == int((raw["language"]=="en").sum()))


---
## Part B — Serve the corpus & ingest it over an API

Real pipelines ingest across a **service boundary**, not a local file read. The FastAPI
app below is **provided** — note the modern **`lifespan`** loader and `app.state.df`
(no module-level global, no deprecated `on_event`). It's also written to
`tools/corpus_api.py` so you can run it as a real server.

In [ ]:

CORPUS_CSV = str(Path("data/corpus_llm.csv").resolve())

@asynccontextmanager
async def lifespan(app: FastAPI):
    df = pd.read_csv(CORPUS_CSV)
    for c in ["body_text","title","section","tags","source_url"]:
        df[c] = df[c].fillna("")                       # NaN -> "" for clean JSON
    app.state.df = df                                  # state on app.state, not a global
    yield

app = FastAPI(title="LLM Corpus API", version="2.0.0", lifespan=lifespan)

@app.get("/health")
async def health():
    return {"ok": True}

@app.get("/v1/corpus")
async def list_docs(page: int = 1, page_size: int = 50,
                    q: str | None = None, language: str | None = None, conf: str | None = None):
    if page < 1 or not (1 <= page_size <= 200):
        raise HTTPException(400, "bad paging params")
    df = app.state.df
    if q:                                              # literal substring, not a regex
        m = (df["body_text"].str.contains(q, case=False, na=False, regex=False)
             | df["title"].str.contains(q, case=False, na=False, regex=False))
        df = df[m]
    if language: df = df[df["language"] == language]
    if conf:     df = df[df["confidentiality"] == conf]
    start = (page - 1) * page_size
    return df.iloc[start:start + page_size].to_dict(orient="records")

Path("tools/corpus_api.py").write_text(
    '# tools/corpus_api.py  —  run:  uvicorn corpus_api:app --port 8009\n'
    'from contextlib import asynccontextmanager\nfrom pathlib import Path\nimport pandas as pd\n'
    'from fastapi import FastAPI, HTTPException\n\n'
    'CORPUS_CSV = str((Path(__file__).parent.parent / "data" / "corpus_llm.csv").resolve())\n\n'
    '@asynccontextmanager\nasync def lifespan(app: FastAPI):\n'
    '    df = pd.read_csv(CORPUS_CSV)\n'
    '    for c in ["body_text","title","section","tags","source_url"]: df[c] = df[c].fillna("")\n'
    '    app.state.df = df\n    yield\n\n'
    'app = FastAPI(title="LLM Corpus API", version="2.0.0", lifespan=lifespan)\n\n'
    '@app.get("/health")\nasync def health(): return {"ok": True}\n')
print("app defined; tools/corpus_api.py written")

In [ ]:

# Provided infra: start the API in a background thread and wait until it answers /health.
PORT = 8009
_server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=PORT, log_level="warning"))
threading.Thread(target=_server.run, daemon=True).start()

API = f"http://127.0.0.1:{PORT}"
def _wait_ready(timeout=20):
    end = time.time() + timeout
    while time.time() < end:
        try:
            if requests.get(f"{API}/health", timeout=0.5).status_code == 200: return True
        except Exception: time.sleep(0.15)
    return False

print("API ready:", _wait_ready(), "->", API)


### B1 — Ingest with pagination

Fetch **every** `public` + `en` document by walking pages until a page comes back empty,
and assemble them into `api_df`. Use `page_size=100`.

> **🧰 Toolbox for Part B** — `requests.get(f"{API}/v1/corpus", params={...})` ·
> `params` keys: `page`, `page_size`, `language`, `conf` · `resp.json()` (a list) ·
> loop until the batch is empty · `pd.DataFrame(rows)`.

In [ ]:

def fetch_all(language=None, conf=None, page_size=100):
    # TODO: page through /v1/corpus until an empty batch; collect rows; return a DataFrame.
    return pd.DataFrame()

api_df = fetch_all(language="en", conf="public")
print("fetched rows:", len(api_df))
api_df.head(3)

In [ ]:

check("B1: API healthy", lambda: requests.get(f"{API}/health", timeout=5).json() == {"ok": True})
check("B1: paginated fetch returns 286 public+en docs", lambda: len(api_df) == 286)
check("B1: API result matches the CSV funnel (same doc_ids)",
      lambda: set(api_df["doc_id"]) == set(raw[(raw["confidentiality"]=="public") & (raw["language"]=="en")]["doc_id"]))


---
## Part C — Clean, filter, de-duplicate

The heart of corpus prep. Normalize text, enforce **governance** + **language** policy,
drop empty/short bodies, then collapse **exact-content duplicates** — keyed on a **content
signature**, not `doc_id` (the planted `DOC-DUP-*` rows share content under new ids).

> **🧰 Toolbox for Part C** — `re.sub(r"\s+", " ", s).strip()` · `Series.map(fn)` ·
> `Series.replace("", ...)` · boolean-mask filtering · `Series.str.len()` ·
> `hashlib.sha1(text.encode()).hexdigest()` · `pd.to_datetime` ·
> `sort_values(...)` + `drop_duplicates(subset=..., keep="first")`.


### C1 — Normalize text

Write `normalize_ws(s)` (collapse all whitespace to single spaces, strip ends, `None`→`""`)
and build `clean`: a copy of `raw` with `title`, `section`, `body_text` normalized and any
empty `section` set to `"Overview"`.

In [ ]:

def normalize_ws(s):
    # TODO: collapse whitespace, strip, None -> "".
    return str(s)

clean = raw.iloc[0:0].copy()   # replace: normalize title/section/body_text on a copy of raw
clean

In [ ]:

check("C1: normalize_ws collapses whitespace + strips", lambda: normalize_ws("  a\t b\n c ") == "a b c")
check("C1: clean has all 1,014 rows with normalized columns",
      lambda: len(clean) == 1014 and clean["body_text"].map(lambda s: "  " not in s).all())


### C2 — Governance + language + min-length

From `clean`, keep only `confidentiality == "public"` **and** `language == "en"`, then drop
bodies shorter than **30** characters. Save the survivors as `kept`, and record the funnel
counts (`n_gov_lang`, `n_kept`, `n_dropped_short`).

In [ ]:

# TODO: filter clean to public & en -> gov_lang; drop body_text shorter than 30 chars -> kept.
#       Record n_gov_lang, n_kept, n_dropped_short.
gov_lang = clean.iloc[0:0].copy()
kept = clean.iloc[0:0].copy()
n_gov_lang = n_kept = n_dropped_short = 0
print(f"public+en: {n_gov_lang}  ->  after min-length: {n_kept}  (dropped {n_dropped_short})")

In [ ]:

check("C2: 286 docs survive governance + language", lambda: n_gov_lang == 286)
check("C2: 5 short/empty bodies dropped", lambda: n_dropped_short == 5)
check("C2: 281 kept after min-length", lambda: n_kept == 281)


### C3 — De-duplicate by content signature

Add a `content_key` = SHA-1 of the normalized, lower-cased `title + " " + body_text`. Keep
the **most recently updated** row per key (parse `updated_at` to datetime, sort descending,
keep first). Save as `dedup`. This is why the planted `DOC-DUP-*` rows collapse though
their `doc_id`s differ.

In [ ]:

def content_key(title, body):
    # TODO: SHA-1 hex of normalized, lower-cased  title + " " + body.
    return ""

# TODO: add content_key, keep newest updated_at per key -> dedup.
dedup = kept.copy()
n_dedup = len(dedup)
print(f"after dedupe: {n_dedup}")

In [ ]:

check("C3: 272 rows after dedupe (removed the 9 planted content-dups)", lambda: n_dedup == 272)
check("C3: no duplicate content_key remains", lambda: dedup["content_key"].duplicated().sum() == 0)
check("C3: keyed on content, not doc_id (all 9 DOC-DUP-* collapsed to <=1)",
      lambda: len(dedup) > 0 and dedup["doc_id"].astype(str).str.startswith("DOC-DUP-").sum() <= 1)


---
## Part D — Write JSONL: RAG chunks + SFT rows

Two products from the same clean corpus. **RAG**: overlapping text chunks with provenance
metadata, one chunk per line. **SFT**: `{"input","output","metadata"}` rows. Then validate
every line and cut a reviewer sample.

> **🧰 Toolbox for Part D** — string slicing for chunks · `orjson.dumps(obj).decode() + "\n"`
> · write one object per line · `json.loads(line)` to validate · `itertools.islice`.
> ⚠️ A chunker that doesn't advance past `overlap` will **loop forever** — always move the
> window forward by `max_chars - overlap`.


### D1 — Overlapping chunker

Write `split_chunks(text, max_chars=300, overlap=60)`: normalize whitespace, return a list
of ≤`max_chars` substrings where each chunk after the first begins `overlap` chars before
the previous chunk ended. Empty text → `[]`; never loop forever.

In [ ]:

def split_chunks(text, max_chars=300, overlap=60):
    # TODO: normalize ws; slice into <=max_chars windows advancing by (max_chars-overlap);
    #       consecutive chunks share `overlap` chars; ""/None -> []; must terminate.
    return []

_probe = split_chunks(dedup["body_text"].iloc[0] if len(dedup) else "")
print("chunks for doc[0]:", len(_probe))

In [ ]:

_long = "x" * 800
_ch = split_chunks(_long, max_chars=300, overlap=60)
check("D1: 800-char text splits into multiple chunks", lambda: len(_ch) >= 3)
check("D1: consecutive chunks overlap by 60 chars",
      lambda: len(_ch) >= 2 and all(_ch[k][-60:] == _ch[k+1][:60] for k in range(len(_ch)-1)))
check("D1: empty text -> no chunks", lambda: split_chunks("") == [])


### D2 — RAG-chunk JSONL with provenance

For every row in `dedup`, chunk `body_text` and write one JSONL line per chunk to
`artifacts/jsonl/rag_chunks.jsonl`: `doc_id`, `chunk_id` (`f"{doc_id}-{j:04d}"`), `text`,
and a `metadata` dict carrying `title, section, tags, source_url, language,
confidentiality, schema_version="rag-chunk-v1"`.

In [ ]:

rag_path = Path("artifacts/jsonl/rag_chunks.jsonl")
n_rag = 0
with rag_path.open("wb") as f:
    # TODO: for each dedup row, write one line per chunk with doc_id, chunk_id, text, metadata.
    pass
print("wrote", n_rag, "chunk lines ->", rag_path)

In [ ]:

_rag = [json.loads(l) for l in rag_path.read_text().splitlines()] if rag_path.exists() else []
check("D2: 546 RAG chunk lines written", lambda: len(_rag) == 546)
check("D2: every line carries doc_id, chunk_id, text, metadata",
      lambda: len(_rag) > 0 and all({"doc_id","chunk_id","text","metadata"} <= set(r) for r in _rag))
check("D2: provenance present (schema_version tag on every chunk)",
      lambda: len(_rag) > 0 and all(r["metadata"].get("schema_version") == "rag-chunk-v1" for r in _rag))


### D3 — SFT/eval JSONL

Take up to **120** docs (`dedup.sample(min(120, len(dedup)), random_state=7)`) and write
`artifacts/jsonl/corpus_sft.jsonl` with rows `{"input","output","metadata"}` — `input` a
prompt referencing the title/section, `output` a short templated answer, `metadata` with
`doc_id`, `type`, `lang`.

In [ ]:

sft_path = Path("artifacts/jsonl/corpus_sft.jsonl")
sample = dedup.sample(min(120, len(dedup)), random_state=7) if len(dedup) else dedup
with sft_path.open("wb") as f:
    # TODO: write {"input","output","metadata"} per sampled row.
    pass
print("wrote", sum(1 for _ in sft_path.open()), "SFT rows ->", sft_path)

In [ ]:

_sft = [json.loads(l) for l in sft_path.read_text().splitlines()] if sft_path.exists() else []
check("D3: 120 SFT rows written", lambda: len(_sft) == 120)
check("D3: every SFT row has input / output / metadata",
      lambda: len(_sft) > 0 and all({"input","output","metadata"} <= set(r) for r in _sft))


### D4 — Validate JSONL & cut a reviewer sample

Write `validate_jsonl(path)` → `(total, bad)` where `bad` counts lines that don't parse as
a JSON **object**. Run it on both files, then write the first **3** lines of each to
`artifacts/samples/jsonl_samples.jsonl` (6 lines total).

In [ ]:

import itertools

def validate_jsonl(path):
    # TODO: return (total, bad); bad = lines that don't parse as a JSON object (dict).
    return (0, 0)

rag_stats = validate_jsonl(rag_path)
sft_stats = validate_jsonl(sft_path)

sample_path = Path("artifacts/samples/jsonl_samples.jsonl")
with sample_path.open("w", encoding="utf-8") as out:
    # TODO: write the first 3 lines of each file (6 total).
    pass
print("rag:", rag_stats, "| sft:", sft_stats)

In [ ]:

check("D4: RAG validates clean (546 total, 0 bad)", lambda: rag_stats == (546, 0))
check("D4: SFT validates clean (120 total, 0 bad)", lambda: sft_stats == (120, 0))
check("D4: reviewer sample has 6 lines", lambda: sum(1 for _ in open(sample_path)) == 6)
score()

In [ ]:

_server.should_exit = True   # release the port; the corpus is now fully materialized to JSONL
print("API stopped; artifacts written under artifacts/")


---
## Wrap-up — answer in this Markdown cell

1. **JSONL vs one JSON array** — three concrete reasons JSONL wins for a large corpus.
2. **Governance + language** — which filters did you apply, and *at what stage* (and why
   there, not later)?
3. **Dedup key** — what did you hash, and why a **content signature** instead of `doc_id`?
4. **RAG vs SFT metadata** — which fields are essential for each output, and why?

**Key takeaways**
- **JSONL is the pipeline lingua franca** — streamable, append-friendly, and one bad line
  doesn't sink the file. A single giant JSON array is an all-or-nothing parse (and an OOM
  risk).
- **Dedupe on content, not identifiers.** `doc_id` is unique by construction; the
  duplicates that hurt an index share *text*. Hash the normalized content.
- **Filter under governance early.** `public`/`en` selection happens *before* you chunk and
  emit — you never want internal or off-language text sitting in a public index.
- **Provenance is not optional.** Every RAG chunk carries `source_url`, `title`, and a
  `schema_version` so a retrieved passage can be traced and re-generated.
- **Validate line-by-line.** The same property that makes JSONL robust — independent
  lines — is what lets you quarantine one bad record instead of failing the batch.
- **Modern FastAPI uses `lifespan`, not `on_event`;** load once at startup, hang state on
  `app.state`.
